<center><img src="../images/DLI Header.png" alt="Header" style="width: 400px;"/></center>

# Hello Camera
### WebCam (USB) Cameras

이 노트북에서는 카메라가 예상대로 젯슨 나노에서 작동하는지 테스트할 수 있습니다.  카메라는 USB 카메라 포트에 이미 연결되어 있어야 합니다.  카메라 렌즈에 필름이나 커버와 같은 장애물이 없는지도 확인하십시오.

<center><img src="../images/usbcam_setup_sm.jpg" width=600/></center>

<div style="border:2px solid black; background-color:#e3ffb3; font-size:12px; padding:8px; margin-top: auto;"><i>
    <h4><i>팁</i></h4>
코드 셀에서 파이썬 또는 시스템 코드를 실행하려면 셀을 선택하고 창 상단에 있는 "RUN" 버튼을 클릭합니다.<br>키보드 단축키: <strong>[SHIFT][ENTER]</strong>
    </i></div>

### Check to see if the device is available
다음 시스템 명령을 실행하여 Jetson Nano의 모든 비디오 장치를 나열합니다. 카메라에 장치 ID가 표시되지 않으면 연결을 확인하십시오. 다음과 유사한 출력 값을 얻어야 합니다.
```text
crw-rw----+ 1 root video 81, 0 Jun  2 17:35 /dev/video0
```

In [1]:
!ls -ltrh /dev/video*

crw-rw---- 1 root video 81, 0 May 11 06:12 /dev/video0


### Create the camera object

먼저 다음 파이썬 코드 셀을 실행하여 라이브러리에서 `USBCamera` 클래스를 가져와 카메라 개체를 만듭니다.`USBCamera`인스턴스는 하나만 만들 수 있습니다. 시스템 비디오 장치를 나열할 때 찾은 올바른 번호로 `capture_device=` 를 설정합니다. `/dev/video0`이 있으면 `capture_device=0`을 설정합니다. 만약 `/dev/video1` 이 있다면 아래의 코드줄에서 `capture_device=1`을 설정하십시오.

In [2]:
from jetcam.usb_camera import USBCamera

#TODO change capture_device if incorrect for your system
camera = USBCamera(width=224, height=224, capture_width=640, capture_height=480, capture_device=0)


(python3.10:465): GStreamer-WARNING **: 06:15:12.982: External plugin loader failed. This most likely means that the plugin loader helper binary was not found or could not be run. You might need to set the GST_PLUGIN_SCANNER environment variable if your setup is unusual. This should normally not be required though.

(python3.10:465): GStreamer-WARNING **: 06:15:12.982: Failed to load plugin '/usr/lib/aarch64-linux-gnu/gstreamer-1.0/libgstnvipcpipeline.so': /usr/lib/aarch64-linux-gnu/gstreamer-1.0/libgstnvipcpipeline.so: file too short
[ WARN:0@0.888] global cap_gstreamer.cpp:1728 open OpenCV | GStreamer warning: Cannot query video position: status=0, value=-1, duration=-1


우리는 `read` 함수를 통해 카메라로부터 프레임을 캡처할 수 있습니다.  

In [3]:
image = camera.read()

print(image.shape)

(224, 224, 3)


`camera` `read` 함수를 호출하는 것은 카메라의 내부 `value`를 업데이트 합니다. 이 값의 `shape`를 살펴봄으로서 우리는 픽셀 높이, 픽셀 넓이, 컬러 채널 숫자를 표현하는 3개의 숫자를 확인할 수 있습니다.

In [4]:
print(camera.value.shape)

(224, 224, 3)


### Create a widget to view the image stream
이 이미지를 노트북에 표시할 "위젯"을 만들 수 있습니다. 이미지를 보려면 이미지를 파란색-녹색-빨간색 형식(brg8)에서 브라우저가 표시할 수 있는 형식(jpeg)으로 변환합니다.

In [5]:
import ipywidgets
from IPython.display import display
from jetcam.utils import bgr8_to_jpeg

image_widget = ipywidgets.Image(format='jpeg')

image_widget.value = bgr8_to_jpeg(image)

display(image_widget)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

모든 것이 올바르게 작동하는 경우 카메라에서 이미지를 볼 수 있습니다. 이미지가 있는 것처럼 보이지만 흐릿하거나 이상한 색이 있으면 렌즈에 보호 필름이나 캡이 없는지 확인하십시오.

이제 카메라의 라이브 스트림을 확인해 보겠습니다. 카메라의 `running` 값은 배경에 있는 값을 연속적으로 업데이트 합니다. 이를 통해 카메라 값 변화에 대한 "callbacks"을 호출할 수도 있습니다.

여기서 "콜백"은 `update_image`라는 함수인데, 아래의 `observe` 메서드를 호출하여 첨부합니다. `update_image`는 처리할 수 있는 새 이미지가 있을 때마다 실행되며, 이 이미지가 위젯에 표시됩니다.

In [6]:
camera.running = True

def update_image(change):
    image = change['new']
    image_widget.value = bgr8_to_jpeg(image)
    
camera.observe(update_image, names='value')

카메라 앞으로 무언가를 움직이면 위젯에서 라이브 비디오 스트림을 볼 수 있습니다. 이를 중지하려면 `unobserve` 메서드를 사용하여 콜백을 부착하지 않습니다.

In [7]:
camera.unobserve(update_image, names='value')

<div style="border:2px solid black; background-color:#e3ffb3; font-size:12px; padding:8px; margin-top: auto;"><i>
    <h4><i>팁</i></h4>
셀을 마우스 오른쪽 단추로 클릭하고 "Create New View for Output"을 선택하여 위젯(또는 셀)을 주피터랩의 새 창 탭으로 이동할 수 있습니다. 이렇게 하면 JupyterLab 노트북을 계속 스크롤하여 카메라 화면을 볼 수 있습니다!
    </i></div>

### Another way to view the image stream
또한 traitlets의 `dlink` 방법을 사용하여 변환을 매개 변수 중 하나로 사용하여 카메라를 위젯에 연결할 수 있습니다. 이렇게 하면 프로세스의 일부 단계를 제거할 수 있습니다

In [8]:
import traitlets

camera_link = traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

`unlink` 함수를 사용하여 카메라/위젯 링크를 제거할 수 있습니다.

In [9]:
camera_link.unlink()

`link`를 활용하여 재연결도 가능합니다.

In [10]:
camera_link.link()

<h1 style="background-color:#76b900;"></h1>

## Before you go...<br><br>카메라 및/또는 노트북 커널을 종료하여 카메라 리소스를 해제합니다.

In [ ]:
# Attention!  Execute this cell before moving to another notebook
# The USB camera application only requires that the notebook be reset
# The CSI camera application requires that the 'camera' object be specifically released

import os
os._exit(00)

다음 지침을 보려면 DLI 과정 페이지로 이동하십시오.

<center><img src="../images/DLI Header.png" alt="Header" style="width: 400px;"/></center>